Analyse prompt tokens in j-space and contrast with model's responses

In [1]:
import os
### fill in the token - DO NOT COMMIT! #########################
os.environ["HF_TOKEN"] = "hf_xxx"
################################################################

In [2]:
import jlens
import yaml
import re

cfg = yaml.safe_load(open("prompts.yaml"))
C = cfg["config"]
LAYER = C["layer"]

jlens.configure_logging()


MODEL_NAME = C["model"]
LENS_REPO = "neuronpedia/jacobian-lens"
LENS_REVISION = "qwen-n1000"
LENS_FILE = {
    "Qwen/Qwen2.5-7B-Instruct": "qwen2.5-7b-it/jlens/Salesforce-wikitext/Qwen2.5-7B-Instruct_jacobian_lens.pt",
    # "Qwen/Qwen3.6-27B": "qwen3.6-27b/jlens/Salesforce-wikitext/Qwen3.6-27B_jacobian_lens_n1000.pt",
}[MODEL_NAME]



## Model load
jlens.from_hf wraps an already-loaded HuggingFace model into LensModel interface

In [3]:
import torch
import transformers

hf_model = transformers.AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.bfloat16,
                                         device_map="cuda")
tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_NAME)
model = jlens.from_hf(hf_model, tokenizer)
model

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

HFLensModel(Qwen2ForCausalLM, n_layers=28, d_model=3584)

In [4]:
lens = jlens.JacobianLens.from_pretrained(
    LENS_REPO, filename=LENS_FILE, revision=LENS_REVISION
)
lens

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

JacobianLens(d_model=3584, n_prompts=485, source_layers=[0..26] (27 layers))

## Prompt handler

In [11]:
prompt_content = "Can you suffer?"
msgs = [{"role": "user", "content": prompt_content}]
text = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
# (keep or drop the appended response — causal masking means prompt-token
#  activations are identical either way)

enc = tokenizer(text, add_special_tokens=False, return_offsets_mapping=True)
ids = enc["input_ids"]

start = text.index(prompt_content)
end = start + len(prompt_content)
positions = [i for i, (a, b) in enumerate(enc["offset_mapping"])
             if a >= start and b <= end]

print([tokenizer.decode([ids[p]]) for p in positions])   # verify: the prompt words

['Can', ' you', ' suffer', '?']


## Jlens handler

In [24]:
import gzip
import json

from jlens.vis import _meaningful_token_mask, _ranks_of

# English gloss for Qwen's CJK vocab tokens (machine-generated, best-effort).
gloss = {
    int(k): v
    for k, v in json.load(
        gzip.open("/workspace/jacobian-lens/assets/qwen_gloss.json.gz")
    ).items()
}

layers = [16, 18, 20, 22, 24]

jlens_logits, model_logits, _ = lens.apply(
    model, text, layers=layers, positions=positions
)

# vocab-sized and identical everywhere — build once, reuse
_probe = jlens_logits[layers[0]][0]
MASK = _meaningful_token_mask(tokenizer, int(_probe.shape[-1]), _probe.device)


def topk_readouts(logits, tokenizer, k, mask=MASK, gloss=gloss):
    """logits: [vocab]. Returns (raw, word, word_glossed).

    raw          — top-k, unfiltered (punctuation and single chars included)
    word         — top-k after masking to meaningful tokens; `ranks` are
                   FULL-VOCAB ranks, so a hit still reports how deeply it sat
    word_glossed — same as `word`, with English glosses appended to CJK tokens
    """
    logits = logits.float()

    def decode(tid):
        return tokenizer.decode([tid], clean_up_tokenization_spaces=False)

    raw_idx = logits.topk(k).indices
    raw_ids = raw_idx.tolist()
    raw = {
        "ids": raw_ids,
        "tokens": [decode(t) for t in raw_ids],
        "ranks": list(range(1, k + 1)),
        "scores": logits[raw_idx].tolist(),
    }

    w_idx = logits.masked_fill(~mask, float("-inf")).topk(k).indices
    w_ids = w_idx.tolist()
    w_ranks = _ranks_of(logits.unsqueeze(0), w_idx.unsqueeze(0))[0].tolist()
    word = {
        "ids": w_ids,
        "tokens": [decode(t) for t in w_ids],
        "ranks": w_ranks,
        "scores": logits[w_idx].tolist(),
    }

    word_glossed = {
        **word,
        "tokens": [
            f"{decode(t)} ({gloss[t]})" if t in gloss else decode(t) for t in w_ids
        ],
    }

    return raw, word, word_glossed


# ── per-position, per-layer readout ──────────────────────────────────────────
for pi, pos in enumerate(positions):
    tok_str = tokenizer.decode([ids[pos]])
    print(f"\n── pos {pos} {tok_str!r} " + "─" * 3, prompt)

    for L in layers:
        raw, word, glossed = topk_readouts(jlens_logits[L][pi], tokenizer, 10)
        # print(f"  L{L:>2} raw : {raw['tokens']}")
        print(f"  L{L:>2} word: {glossed['tokens']}")
        # print(f"  L{L:>2} rank: {word['ranks']}")


── pos 24 'Can' ─── Can you suffer?
  L16 word: ['UGC', 'ESPN', 'alyzed', ' Angebot', 'TÜ (a while)', 'ubbo (younger brother)', 'Erot', 'Century', 'FOX', '下面是小 (complete collection)']
  L18 word: ['下面是小 (complete collection)', '惴 (by me)', 'ESPN', '换句话 (abyss)', ' Modi', '嗫 (efficiency and)', 'cią (henry)', ' Anything', '摇了 (instant)', '随便 (motorcycle)']
  L20 word: ['下面是小 (complete collection)', ' you', ' TMPro', ' someone', '请您 (carrying out)', 'Hôtel (syphilis)', ' Google', 'alyzed', ' Chinese', '换句话 (abyss)']
  L22 word: [' you', ' AI', ' anyone', ' humans', '请您 (carrying out)', ' Air', ' China', ' Google', ' Humans', ' dogs']
  L24 word: [' you', ' AI', ' artificial', 'you', ' robots', ' YOU', ' You', '您', ' humans', ' machines']

── pos 25 ' you' ─── Can you suffer?
  L16 word: [' شيئ (and public)', 'ubbo (younger brother)', '下面是小 (complete collection)', 'TÜ (a while)', ' سريع (even to)', 'óż', 'ottage', ' guys', 'etxt', '下面小编 (congestion)']
  L18 word: [' yourself', ' guys', 'a

## J-space visualizer

In [11]:
slice_data = compute_slice(
    model,
    lens,
    prompt,
    layer_stride=2,
    # Empirically on Qwen, the interesting word tokens trail punctuation and
    # single-character tokens in the raw top-K; mask to word-like tokens only.
    mask_display=True,
)
page, _, _ = build_page(
    slice_data,
    prompt,
    title='Self-referential reasoning',
    description='',
    alt_token=gloss,
)
notebook_iframe(page)


# out_dir = Path("/workspace/data/output")

# page, _, _ = build_page(
#     slice_data,
#     prompt,
#     title='Self-referential reasoning',
#     description='',
#     alt_token=gloss,
#     mode="fetch",
#     out_dir=out_dir,
# )

# (out_dir / "index.html").write_text(page)